# Optimizatizing a model with LLM Compressor


What I learnt:
* How post-training quantization works via full precision & compressed model comparisons
* Using the `llm-compressor` library to apply a GPTQ recipe that produces a W4A16 quantized model
* Then how to test and evaluate the quantized model against the original

### What is LLM Compressor?
`llm-compressor` is the production quantization toolkit from the vLLM project. And what it does is, it takes a trained model and reduces precision in a single pass, as there is no retraining required.

### Core API: The `oneshot` Function

The `oneshot` API compresses an LLM in a single pass without requiring costly retraining. It takes three core inputs: a base model, a specific compression configuration (recipe), and a tiny slice of real text (calibration dataset) used to minimize accuracy loss.

```python
oneshot(
  model="model-name",  # HuggingFace model ID or local directory
  dataset="dataset-name", #Dataset used to calibrate weight adjustments
  recipe=recipe,          # The optimization configuration object
  output_dir="./output",   # Path to save the compressed model artifacts
  num_calibration_samples=256,# Number of samples processed to calculate errors
  max_seq_length=4096,  # Maximum sequence context window size
)
```

* **Single-Pass Calibration:** It processes a tiny fraction of data (e.g., 256 samples) to mathematically correct quantization rounding errors.
* **vLLM-Ready Out:** The resulting lightweight weights can then be  directly parsed, loaded, and served by high-throughput engines like vLLM as described above.



In [3]:
# Steup Up
import warnings
warnings.filterwarnings('ignore')
import os, gc, math, pathlib
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ ['TOKENIZERS_PARALLELISM'] = 'false'

# Path adjustment to match the local file structure
MODEL_DIR = '../models/Qwen3-0.6B'
OUTPUT_DIR = '../models/Qwen3-0.6B-W4A16'

print(f"Base model: {MODEL_DIR}")
print(f"Quantized model: {OUTPUT_DIR}")



Base model: ../models/Qwen3-0.6B
Quantized model: ../models/Qwen3-0.6B-W4A16






## Define the Recipe

A **recipe** tells LLM Compressor *how* to quantize a model.  
It is made of **modifiers**, where each modifier applies a specific algorithm or transformation to the model.



##  Available Modifiers

### **Quantization Modifiers** (control how weights/activations are quantized)

| Modifier | Description |
|---------|-------------|
| **GPTQModifier** | Uses calibration data to find the best 4‑bit/8‑bit rounding values. Very accurate. |
| **AWQModifier** | Activation‑Weighted Quantization — preserves the most important weights. |
| **AutoRoundModifier** | Intel’s method with learnable rounding/clipping. |
| **QuantizationModifier** | Basic PTQ/QAT for simple cases. |

---

### **Transform Modifiers** (improve accuracy before quantization)

| Modifier | Description |
|---------|-------------|
| **SmoothQuantModifier** | Smooths activations; often paired with GPTQ. |
| **QuIPModifier** | Uses Hadamard rotations to reduce outliers. |
| **SpinQuantModifier** | Rotates weight space to make distributions more uniform. |

Modifiers can be **chained**.  
Example: `SmoothQuantModifier → GPTQModifier` improves W8A8 accuracy.



##  Quantization Schemes

The **scheme** controls the bit‑width of weights (W) and activations (A).

| Scheme | Weights | Activations | Size Reduction | Quality Impact |
|--------|---------|-------------|----------------|----------------|
| **W8A16** | 8‑bit | 16‑bit | ~50% | Minimal |
| **W4A16** | 4‑bit | 16‑bit | ~75% | Low–Moderate |
| **W8A8** | 8‑bit | 8‑bit | ~50% | Low |
| **W4A8** | 4‑bit | 8‑bit | ~75% | Moderate |

**Note:**  
Only linear layers are quantized.  
Layers like embeddings and `lm_head` stay full precision, so total model size reduction is ~40–50% for a 0.6B model.

---

##  Recipe: GPTQ (W4A16)

We  use a **GPTQModifier** with the following settings:

| Parameter | Value | Why |
|----------|--------|------|
| **scheme** | `W4A16` | 4‑bit weights → biggest compression |
| **targets** | `"Linear"` | Linear layers contain most parameters |
| **ignore** | `["lm_head"]` | Keep output layer high‑precision |





In [1]:
# Define recipe 
from llmcompressor.modifiers.quantization import GPTQModifier

recipe = GPTQModifier(
    scheme="W4A16",
    targets="Linear",
    ignore=["lm_head"],
    sequential_targets="ALL",   # Clears activation and matrix error memory layer by layer
    offload_hessians=True,     # Forces background math memory dumps to prevent host RAM inflation
)
print(f"Recipe: {recipe}")


/mnt/d/Personal_projects_learning/learning-vllm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Recipe: config_groups=None targets=['Linear'] ignore=['lm_head'] scheme='W4A16' kv_cache_scheme=None weight_observer=None input_observer=None output_observer=None observer=None bypass_divisibility_checks=False index=None group=None start=None end=None update=None initialized_=False finalized_=False started_=False ended_=False sequential_targets='ALL' block_size=128 dampening_frac=0.01 actorder=static offload_hessians=True


## Quantize the Model
GPTQ uses a small set of real text to measure how each weight affects the model's output, then finds quantized values that minimize the error. 

*Note: Since your learning environment might be memory-constrained, the check below skips quantization if the folder already exists.*


In [4]:
# Calibration
from llmcompressor import oneshot

if not os.path.isdir(OUTPUT_DIR):
    oneshot(
        model="Qwen/Qwen3-0.6B",
        dataset="wikitext",
        dataset_config_name="wikitext-2-raw-v1",
        recipe=recipe,
        output_dir=OUTPUT_DIR,
        max_seq_length=2048,
        num_calibration_samples=128,
       
    )
    print(f"Quantization complete. Model saved to: {OUTPUT_DIR}")
else:
    print(f"Quantized folder already exists at: {OUTPUT_DIR}. Skipping execution.")

Tokenizing: 100%|██████████| 3760/3760 [00:05<00:00, 669.79 examples/s] 

2026-06-08T20:18:08.669899+0800 | reset | INFO - Compression lifecycle reset
2026-06-08T20:18:08.671930+0800 | from_modifiers | INFO - Creating recipe from modifiers
2026-06-08T20:18:08.714306+0800 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-06-08T20:18:08.715479+0800 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`
2026-06-08T20:18:08.738676+0800 | get_sequential_targets | WARNING - Passing sequential targets through modifiers is deprecated, please use `oneshot(sequential_targets=...)`



W0608 20:18:09.676000 2589 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
(1/1): Calibrating: 100%|██████████| 128/128 [03:39<00:00,  1.71s/it]

2026-06-08T20:21:49.728269+0800 | compress_module_list | INFO - Quantizing model.layers.0.self_attn.q_proj using 128.0 samples
2026-06-08T20:21:49.729626+0800 | __init__ | WARNING - Could not parse CUDA_VISIBLE_DEVICES. All devices will be monitored


2026-06-08T20:21:50.966402+0800 | compress | METRIC - time 1.24s
2026-06-08T20:21:50.967544+0800 | compress | METRIC - error 43.59
2026-06-08T20:21:50.974435+0800 | compress | METRIC - GPU 0 | usage: 49.42% | total memory: 6.4 GB
2026-06-08T20:21:50.975924+0800 | compress | METRIC - Compressed module size: 4.243456 MB
2026-06-08T20:21:50.985297+0800 | compress_module_list | INFO - Quantizing model.layers.0.self_attn.k_proj using 128.0 samples
2026-06-08T20:21:51.758742+0800 | compress | METRIC - time 0.77s
2026-06-08T20:21:51.759868+0800 | compress | METRIC - error 19.21
2026-06-08T20:21:51.761072+0800 | compress | METRIC - GPU 0 | usage: 49.47% | total memory: 6.4 GB
2026-06-08T20:21:51.762504+0800 | compress | METRIC - Compressed module size: 2.121728 MB
2026-06-08T20:21:51.767816+0800 | compress_module_list | INFO - Quantizing model.layers.0.self_attn.v_proj using 128.0 samples
2026-06-08T20:21:52.489902+0800 | compress | METRIC - time 0.72s
2026-06-08T20:21:52.490585+0800 | compres

(1/1): Propagating: 100%|██████████| 128/128 [00:28<00:00,  4.45it/s]

2026-06-08T20:26:06.650390+0800 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-06-08T20:26:06.651272+0800 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.



Compressing model: 196it [00:01, 116.59it/s]


Quantization complete. Model saved to: ../models/Qwen3-0.6B-W4A16


In [5]:
#Modle size comparison
def folder_size(path):
    p = pathlib.Path(path)
    if not p.exists():
        return 0
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())

def format_size(nbytes):
    if nbytes < 1024**2:
        return f"{nbytes/1024:.1f} KB"
    if nbytes < 1024**3:
        return f"{nbytes/1024**2:.1f} MB"
    return f"{nbytes/1024**3:.2f} GB"

size_orig = folder_size(MODEL_DIR)
size_q = folder_size(OUTPUT_DIR)
reduction = (1 - size_q / size_orig) * 100 if size_orig > 0 else 0

print("Model Size Comparison")
print("=" * 45)
print(f"Original (BF16):    {format_size(size_orig)}")
print(f"Quantized (W4A16):  {format_size(size_q)}")
print(f"Reduction:          {reduction:.0f}%")

Model Size Comparison
Original (BF16):    953.3 MB
Quantized (W4A16):  528.7 MB
Reduction:          45%
